In [ ]:
import os
import warnings
import logging
import random

import torch
import pandas as pd
import numpy as np
from rdkit import Chem
from tqdm import tqdm
import numpy as np

from bionemo.utils.hydra import load_model_config
from bionemo.model.molecule.megamolbart.infer import MegaMolBARTInference

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

logging.basicConfig(level=logging.INFO)
logging.getLogger("nemo_logger").setLevel(logging.ERROR)

In [ ]:
bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

In [ ]:
#!python download_artifacts.py --model_dir ${BIONEMO_HOME}/models --models megamolbart

In [ ]:
%%capture --no-display --no-stderr cell_output

# Load pre-trained model checkpoints
checkpoint_path = f"{bionemo_home}/models/molecule/megamolbart/megamolbart.nemo"

# Load starting config for MolMIM inference
cfg = load_model_config(config_name="megamolbart_infer.yaml", config_path=f"{bionemo_home}/examples/tests/conf/")

# Point YAML configuration file to the location of the desired checkpoints
cfg.model.downstream_task.restore_from_path = checkpoint_path
#cfg.model.encoder.hidden_steps = 2

# Create model object based on desired configuration
model = MegaMolBARTInference(cfg, interactive=True)

In [ ]:
expert_smiles_path = "/workspace/bionemo/data/data_experts_1.csv"
expert_smiles_filename = expert_smiles_path.split("/")[-1]
expert_smiles = pd.read_csv(expert_smiles_path)['smiles'].to_list()

In [ ]:
# Defining the chemical sampling/generation function
def chem_sample(
        reference_smiles: list[str],
        num_samples: int = 100, # Maximum number of generated molecules per query compound
        scaled_radius: float = 1.0, # Radius of exploration [range: 0.0 - 1.0] --- the extent of perturbation of the original hidden state for sampling
) -> list:

    hidden_states, enc_masks = model.seq_to_hiddens(reference_smiles)    # Obtaining the hidden state representation(s) for input SMILES
    sample_masks = enc_masks.repeat_interleave(num_samples, 0)    # This and following lines are perturbing the hidden state to obtain analogous compounds
    perturbed_hiddens = hidden_states.repeat_interleave(num_samples, 0)
    perturbed_hiddens = perturbed_hiddens + (scaled_radius * torch.randn(perturbed_hiddens.shape).to(perturbed_hiddens.device))
    samples = model.hiddens_to_seq(perturbed_hiddens, sample_masks)

    # In this code block, we are doing some checks for the validity of the generated SMILES and for de-duplication of the set,
    #    returning only valid and unique SMILES
    samples = set(samples)
    valid_molecules = []
    for smi in set(samples):
        mol = Chem.MolFromSmiles(smi)
        if mol:
            valid_molecules.append(smi)
    uniq_canonical_smiles = [Chem.MolToSmiles(Chem.MolFromSmiles(smi),True) for smi in valid_molecules]

    return uniq_canonical_smiles

In [ ]:
num_samples: int = 100
scaled_radius: float = 1.0

## Iterative generation - for GPU-poor people

In [ ]:
gen_smis_lst = []
for sml in tqdm(expert_smiles, total=len(expert_smiles)):
    gen_smis = chem_sample([sml], num_samples=num_samples, scaled_radius=scaled_radius)
    gen_smis_lst.extend(gen_smis)

## Equal chunk generation - for PSNC people

In [ ]:
# gen_smis_lst = []
# # n = 1 # CUDA OOM
# n = 2
# for chunk in tqdm(np.array_split(expert_smiles, n), total=n):
#     gen_smis = chem_sample(chunk, num_samples=num_samples, scaled_radius=scaled_radius)
#     gen_smis_lst.extend(gen_smis)

In [ ]:
len(gen_smis_lst)

In [ ]:
def format_kwargs(kwargs: dict) -> str:
    return "_".join([f"{k}_{v}" for k, v in kwargs.items()])

In [ ]:
os.makedirs("data/outputs", exist_ok=True)

fine_tuned_str = ""

# save to .csv
pd.DataFrame(
    data={"SMILES": gen_smis_lst}
).to_csv(f"data/outputs/bionemo_megamolbart_{expert_smiles_filename.split('.')[0]}_{fine_tuned_str}_num_samples_{num_samples}_scaled_radius_{scaled_radius}.csv", index=False)